# WildSpeech M8.3 — Performance Bottleneck Audit
**GitHub-first notebook.** Loads the checked-out M8.3 source directly from the WildSpeech repository. No ZIP upload and no raw audio extraction. Outputs remain private in Google Drive; synthetic tests are not competition validation.

In [ ]:
from pathlib import Path
import subprocess, sys
IN_COLAB = Path("/content").is_dir()
ROOT = Path("/content/WildSpeech_git") if IN_COLAB else Path.cwd() / "WildSpeech_git"
REPO = ROOT / "WildSpeech_M8"
URL = "https://github.com/mybenkhadda/WildSpeech.git"
if not (REPO / "pyproject.toml").is_file():
    if ROOT.exists():
        raise RuntimeError(f"{ROOT} exists without the M8 package. Select an empty ROOT.")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", URL, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "origin", "main"], check=True)
assert (REPO / "src/wildspeech_m8/m83_bottleneck.py").is_file(), "M8.3 not found: update your GitHub checkout."
print("Repository:", ROOT)
print("M8.3 available:", (REPO / "src/wildspeech_m8/m83_bottleneck.py").is_file())

## Install and import M8.3 from GitHub
The repository already contains the M8.3 module. This install works without ZIP extraction or manually copying code.

In [ ]:
import importlib
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)], check=True)
srcdir = str(REPO / "src")
if srcdir not in sys.path:
    sys.path.insert(0, srcdir)
importlib.invalidate_caches()
from wildspeech_m8.m83_bottleneck import AuditConfig, run_official
print("M8.3 module imported from:", REPO / "src/wildspeech_m8/m83_bottleneck.py")

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/WildSpeech"  # edit for your official data
OUTPUT_ROOT = "/content/drive/MyDrive/WildSpeech_M8"
PROFILE_CSV = ""  # optional: existing M7 profile file
SMOKE = True      # switch to False for exhaustive analysis
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
assert Path(DATA_ROOT).is_dir(), f"Official WildSpeech directory missing: {DATA_ROOT}"
assert (Path(DATA_ROOT)/"train_labels.csv").is_file(), "train_labels.csv not found in DATA_ROOT"
print("Data:", DATA_ROOT, "Output:", OUTPUT_ROOT, "Smoke:", SMOKE)

In [ ]:
cfg = AuditConfig(
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    profile_csv=PROFILE_CSV,
    quick=SMOKE,
    folds=3 if SMOKE else 5,
    repeats=1 if SMOKE else 2,
    bootstrap=50 if SMOKE else 400,
    seed=2026,
)
result = run_official(cfg)
print("Rows:", result["rows"], "Schools:", result["schools"], "Features:", result["feature_count"])
print("Diagnostic summary:")
for note in result["notes"]:
    print(" •", note)

In [ ]:
from IPython.display import display, Markdown, Image
import pandas as pd
folder = Path(OUTPUT_ROOT) / "m83_bottleneck"
report = folder / "M83_BOTTLENECK_REPORT.md"
assert report.is_file(), f"Missing report: {report}"
display(Markdown(report.read_text(encoding="utf8")))
for table in ["fold_metrics", "paired_bootstrap", "residual_probe", "negative_control", "learning_curve"]:
    p = folder / "tables" / (table + ".csv")
    if p.exists():
        print("TABLE:", table)
        display(pd.read_csv(p).head(25))
for fig in sorted((folder / "figures").glob("*.png")):
    print(fig.name)
    display(Image(filename=str(fig), width=860))

## Run the full study
After checking the smoke results, set `SMOKE = False` in the configuration cell and rerun the execution and report cells. The full study includes group-aware CV and residual diagnostics. Do not publish private student-level predictions or school identifiers.